# TAHMO DataHub API wrapper: examples

Run this notebook from the project root (the folder that contains `TAHMO/`).
Install the dependencies first with `pip install -r requirements.txt`.

## 1. Set up and credentials

The secret is typed hidden, so it is not stored in the notebook.

In [ ]:
import getpass
import pandas as pd
from TAHMO import apiWrapper

api = apiWrapper()
api.setCredentials(input('API key: ').strip(), getpass.getpass('API secret: '))

## 2. Explore variables and stations

In [ ]:
variables = api.getVariables()
for shortcode, variable in list(variables.items())[:10]:
    print(shortcode, '-', variable.get('description', ''))

In [ ]:
stations = api.getStations()
print(len(stations), 'stations')

# Look at one record to see which fields (and coordinate names) it has.
next(iter(stations.values()))

## 3. One station

`variables` is a list of shortcodes (leave it out for all variables). An end date without a time includes the whole day.

In [ ]:
df = api.getMeasurements('TA00001',
                         startDate='2024-01-01',
                         endDate='2024-01-31',
                         variables=['pr', 'te'])
df.head()

Raw (unfiltered) data uses `getRawMeasurements`, or `dataset='raw'`:

In [ ]:
raw = api.getRawMeasurements('TA00001', '2024-01-01', '2024-01-07', ['te'])
raw.head()

## 4. Several stations

The columns become a MultiIndex of (station, variable).

In [ ]:
multi = api.getMeasurementsForStations(['TA00001', 'TA00002'],
                                       '2024-01-01', '2024-01-31',
                                       variables=['pr'])
print(api.lastErrors)   # stations that failed, if any
multi.head()

## 5. Stations inside a latitude/longitude bounding box

Roughly Kenya: latitude -4.7 to 5.0, longitude 33.9 to 41.9.

In [ ]:
in_box = api.getStationsInBounds(latMin=-4.7, latMax=5.0, lonMin=33.9, lonMax=41.9)
print(len(in_box), 'stations:', list(in_box)[:10])

In [ ]:
# Start small: a short period keeps the number of API calls low.
box_df = api.getMeasurementsInBounds(latMin=-4.7, latMax=5.0, lonMin=33.9, lonMax=41.9,
                                     startDate='2024-01-01', endDate='2024-01-07',
                                     variables=['pr'])
box_df.shape

## 6. Save as CSV, Parquet or Zarr

The format comes from the extension. Zarr is written as a folder.

In [ ]:
api.saveData(box_df, 'output/kenya_rainfall.csv')
api.saveData(box_df, 'output/kenya_rainfall.parquet')
api.saveData(box_df, 'output/kenya_rainfall.zarr')

## 7. Read the files back

In [ ]:
import xarray as xr

from_csv = pd.read_csv('output/kenya_rainfall.csv', index_col='time', parse_dates=True)
from_parquet = pd.read_parquet('output/kenya_rainfall.parquet')
from_zarr = xr.open_zarr('output/kenya_rainfall.zarr').to_dataframe()
from_parquet.head()